<a href="https://colab.research.google.com/github/leestorm4520/ArtificialIntelligence_UbiquantMarketPrediction/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DEPENDENCIES

pip3 install dash
pip3 install dash-html-components
pip3 install dash-core-components

In [ ]:
# IMPORTS

import dash
import dash_core_components as dcc
import dash_html_components as html
import pandas as pd
import plotly.graph_objs as go
from dash.dependencies import Input, Output
from keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [ ]:
# CODE

app = dash.Dash()
server = app.server

scaler = MinMaxScaler(feature_range=(0,1))

# Import stock data here
DF_aadr_us = pd.read_csv(fileLocation, delimiter = ',')

DF_aadr_us["Date"] = pd.to_datetime(DF_aadr_us.Date, format="%Y-%m-%d")
DF_aadr_us.index = DF_aadr_us['Date']

data = DF_aadr_us.sort_index(ascending=True, axis=0)
new_data = pd.DataFrame(index=range(0,len(DF_aadr_us)), columns=['Date', 'Close'])

for i in range(0, len(data)):
    new_data["Date"][i] = data['Date'][i]
    new_data["Close"][i] = data["Close"][i]

new_data.index = new_data.Date
new_data.drop("Date", axis=1, inplace=True)

dataset = new_data.values

# Split dataset into 2 halves
set_size = int(len(dataset) / 2)

train = dataset[0:set_size, :]
valid = dataset[set_size:, :]

scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(dataset)

x_train, y_train = [], []

for i in range(60, len(train)):
    x_train.append(scaled_data[i-60:i,0])
    y_train.append(scaled_data[i,0])

x_train, y_train = np.array(x_train), np.array(y_train)

x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

# Load model